# Tuning hyperparameters and handling imbalanced classes

Machine Learning Engineer track, lesson MLE4. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** search settings with cross-validation, report a tuned model next to its baseline on a test part scored once, compare class weights with a change of threshold, and resample only inside the folds.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers
import os
import subprocess
import sys
import tempfile
import urllib.request

import numpy as np
import pandas as pd

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 2026       # the seed of the whole track

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"

con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")

# Known answer from the dataset README: 21,841 people and 177,379 events.
assert con.execute("SELECT (SELECT COUNT(*) FROM users), (SELECT COUNT(*) FROM events)").fetchall()[0] == (21841, 177379)

# The modeling table of the Data Scientist track: one row per person, built by the query of lesson DS3, "Building a modeling table in SQL".
# Run this cell as it is; DS3 explains it line by line.
MODELING_SQL = """WITH people AS (            -- one row per person: the spine of the table
  SELECT u.user_id, u.signup_date, u.channel, u.region, s.platform,
         s.event_ts AS signup_ts,
         s.event_ts + INTERVAL 7 DAY AS cutoff,       -- the prediction moment
         s.event_ts + INTERVAL 30 DAY AS horizon      -- the end of the label window
  FROM users AS u
  JOIN events AS s ON s.user_id = u.user_id AND s.event_name = 'signup_complete'
  WHERE u.signup_date BETWEEN DATE '2026-01-01' AND DATE '2026-05-31'
),
clean AS (                  -- retry copies removed: keep the first copy of each event
  SELECT *
  FROM events
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, event_ts, event_name, platform ORDER BY received_at, event_id) = 1
),
known AS (                  -- what the app knew at the cutoff: it had happened and it had arrived
  SELECT p.user_id, c.event_name, c.event_ts, p.cutoff
  FROM people AS p
  JOIN clean AS c ON c.user_id = p.user_id
  WHERE c.event_ts <= p.cutoff AND c.received_at <= p.cutoff
),
features AS (
  SELECT user_id,
         COUNT(*) FILTER (WHERE event_name = 'session_start') AS sessions_7d,
         COUNT(*) FILTER (WHERE event_name = 'add_expense') AS expenses_7d,
         COUNT(*) FILTER (WHERE event_name IN ('session_start', 'add_expense') AND event_ts > cutoff - INTERVAL 3 DAY) AS recent_3d,
         COUNT(DISTINCT CAST(event_ts AS DATE)) FILTER (WHERE event_name IN ('session_start', 'add_expense')) AS active_days_7d,
         CAST(COUNT(*) FILTER (WHERE event_name = 'onboarding_complete') > 0 AS INTEGER) AS onboarded,
         CAST(COUNT(*) FILTER (WHERE event_name IN ('view_paywall', 'paywall_viewed')) > 0 AS INTEGER) AS saw_paywall,
         CAST(COUNT(*) FILTER (WHERE event_name = 'purchase') > 0 AS INTEGER) AS bought_plus
  FROM known
  GROUP BY user_id
),
label AS (                  -- from the other side of the cutoff: no visit and no expense in days 8 to 30
  SELECT p.user_id,
         CAST(NOT EXISTS (SELECT 1 FROM events AS e
                          WHERE e.user_id = p.user_id AND e.event_name IN ('session_start', 'add_expense')
                            AND e.event_ts > p.cutoff AND e.event_ts <= p.horizon) AS INTEGER) AS churned
  FROM people AS p
)
SELECT p.user_id, p.signup_date, p.channel, p.platform, p.region,
       f.sessions_7d, f.expenses_7d, f.recent_3d, f.active_days_7d, f.onboarded, f.saw_paywall, f.bought_plus,
       l.churned
FROM people AS p
JOIN features AS f ON f.user_id = p.user_id
JOIN label AS l ON l.user_id = p.user_id
ORDER BY p.user_id"""
table = con.sql(MODELING_SQL).df()
print(f"Data loaded: {len(table):,} people, {int(table['churned'].sum()):,} of them churned.")

## A table with a rare outcome, and its split

In the real table about 54 percent of the people churned, which is not rare. To practice with a rare outcome, this cell keeps every person who stayed and about 1 in 10 of the people who churned, drawn with seed 2026. The numbers of this lesson describe that made-up sample, not the app. The split is stratified, and the test part is scored once, in unit 2. Run this cell as it is.

In [ ]:
STATIC = ["channel", "platform", "region"]                    # known at signup
WEEK1 = ["sessions_7d", "expenses_7d", "recent_3d", "active_days_7d", "onboarded", "saw_paywall", "bought_plus"]    # known at day 7
rng = np.random.default_rng(SEED)
keep = (table["churned"] == 0) | (rng.random(len(table)) < 0.1)       # everyone who stayed, and about 1 in 10 of the churners
rare = table[keep]
X, y = rare[STATIC + WEEK1], rare["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
folds = StratifiedKFold(5, shuffle=True, random_state=SEED)
cat_columns = make_column_selector(dtype_exclude="number")
num_columns = make_column_selector(dtype_include="number")


def build(model):
    """One-hot text columns, scaled number columns, then the model."""
    prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat_columns), ("num", StandardScaler(), num_columns)])
    return Pipeline([("prep", prep), ("model", model)])


print(len(rare), round(float(y.mean()), 3), len(X_train), len(X_test))

## Unit 1: A search is a loop over settings

**Predict first.** The grid below has 8 settings: 4 of logistic regression (`C` of 0.01 or 1, with or without class weights) and 4 of boosted trees. Each is scored by average precision on the same five folds. Will the best setting beat the plain logistic regression (`C` of 1, no weights) by more than the spread of the best setting's own five fold scores (their standard deviation)?

In [ ]:
grid = [{"model": [LogisticRegression(max_iter=1000)], "model__C": [0.01, 1.0], "model__class_weight": [None, "balanced"]},
        {"model": [HistGradientBoostingClassifier(early_stopping=False, random_state=SEED)],
         "model__learning_rate": [0.05, 0.2], "model__max_depth": [2, 4]}]
search = GridSearchCV(build(LogisticRegression()), grid, scoring="average_precision", cv=folds)
search.fit(X_train, y_train)                                  # 8 settings x 5 folds = 40 fits, then one refit of the winner on all the training rows
results = pd.DataFrame(search.cv_results_)
best = results.loc[search.best_index_]
plain = results[(results["param_model__C"] == 1.0) & results["param_model__class_weight"].isna()].iloc[0]
print(len(results), type(search.best_estimator_["model"]).__name__)
print(round(float(best["mean_test_score"]), 3), round(float(best["std_test_score"]), 3), round(float(plain["mean_test_score"]), 3))

## Unit 2: A baseline, and a test part scored once

**Predict first.** The flat guess gives every person the same chance. What average precision will `DummyClassifier(strategy="prior")` score on the test part, and about what will the tuned model score there?

In [ ]:
flat = DummyClassifier(strategy="prior").fit(X_train, y_train)
tuned = search.best_estimator_                                # already refitted on all the training rows
for label, model in (("flat guess", flat), ("tuned", tuned)):
    print(label, round(float(average_precision_score(y_test, model.predict_proba(X_test)[:, 1])), 3))
print(round(float(y_test.mean()), 3))

## Unit 3: Class weights or a threshold

**Predict first.** A plain logistic regression and the same model with `class_weight="balanced"` are scored on a validation part cut from the training rows, at the usual cut of 0.5 (a person is flagged when the chance is at least 0.5). What share of the churners will each catch (recall)? And when the plain model's cut is moved until its recall equals the balanced model's, will its precision be clearly lower than, about the same as, or clearly higher than the balanced model's?

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=SEED, stratify=y_train)
plain_lr = build(LogisticRegression(max_iter=1000)).fit(X_fit, y_fit)
balanced_lr = build(LogisticRegression(max_iter=1000, class_weight="balanced")).fit(X_fit, y_fit)
chances = {"plain": plain_lr.predict_proba(X_val)[:, 1], "balanced": balanced_lr.predict_proba(X_val)[:, 1]}
for label, p in chances.items():
    flagged = p >= 0.5
    print(label, round(float(recall_score(y_val, flagged)), 3), round(float(precision_score(y_val, flagged, zero_division=0)), 3))
target = recall_score(y_val, chances["balanced"] >= 0.5)       # the recall the balanced model has at 0.5
cut = max(t for t in np.unique(chances["plain"]) if recall_score(y_val, chances["plain"] >= t) >= target)
print(round(float(cut), 3), round(float(recall_score(y_val, chances["plain"] >= cut)), 3), round(float(precision_score(y_val, chances["plain"] >= cut)), 3))

## Unit 4: Resample inside the folds

**Predict first.** Each churner in the training rows is copied 5 times. In the first route the copies are made before the five folds are cut, so a person's copies can land in both a fitting part and a scoring part; in the second route each fold's fitting rows are copied after the cut and the scoring rows are left as they are. Will the second route score higher than, equal to or lower than the first on average precision?

In [ ]:
def copies(X_part, y_part, times=5):
    """Every churner `times` times, every person who stayed once."""
    extra = X_part[y_part == 1]
    return pd.concat([X_part] + [extra] * (times - 1)), np.concatenate([y_part] + [y_part[y_part == 1]] * (times - 1))


X_big, y_big = copies(X_train, y_train.to_numpy())              # copies made before the folds
before = cross_val_score(build(LogisticRegression(max_iter=1000)), X_big, y_big, cv=folds, scoring="average_precision")
inside = []
for fit_ix, score_ix in folds.split(X_train, y_train):         # copies made after the cut, in the fitting rows only
    X_f, y_f = copies(X_train.iloc[fit_ix], y_train.iloc[fit_ix].to_numpy())
    model = build(LogisticRegression(max_iter=1000)).fit(X_f, y_f)
    inside.append(average_precision_score(y_train.iloc[score_ix], model.predict_proba(X_train.iloc[score_ix])[:, 1]))
print(round(float(before.mean()), 3), round(float(np.mean(inside)), 3))

## Finish it

### Exercise 1: Finish it

**Task:** **Does the metric choose the setting?** Run the same grid with `scoring="roc_auc"` instead of average precision. Put in `answer_1` the list [`C` of the best logistic setting by ROC AUC, `True` if it is the same `C` as the best logistic setting by average precision]: both read from `cv_results_`, restricted to the four logistic rows, rounded to 3 decimals for scores and left as the grid value for `C`. To keep it short, search only the logistic part, `grid[0]`.

**Expected:** `answer_1` is a list of a number and a boolean.

In [ ]:
logistic_grid = grid[0]                                       # given: the four logistic settings
# search it twice, once with scoring="average_precision" and once with scoring="roc_auc", on the five folds of the lesson
# then compare the C of the best setting of each
answer_1 = None  # replace None with [C of the best setting by ROC AUC, True or False]

In [ ]:
_ = check("Exercise 1", "answer_1", [1.0, True], traps=[([0.01, False], "Read best_params_ of the search, not the first row.")])

<details>
<summary>Hint</summary>

`GridSearchCV(build(LogisticRegression()), logistic_grid, scoring=..., cv=folds).fit(X_train, y_train)`; the winner is in `best_params_["model__C"]`.

</details>

<details>
<summary>Half-solution</summary>

```python
by_ap = GridSearchCV(build(LogisticRegression()), logistic_grid, scoring="average_precision", cv=folds).fit(X_train, y_train)
by_auc = ...
answer_1 = [by_auc.best_params_["model__C"], ...]
```

</details>

<details>
<summary>Solution</summary>

```python
logistic_grid = grid[0]
by_ap = GridSearchCV(build(LogisticRegression()), logistic_grid, scoring="average_precision", cv=folds).fit(X_train, y_train)
by_auc = GridSearchCV(build(LogisticRegression()), logistic_grid, scoring="roc_auc", cv=folds).fit(X_train, y_train)
answer_1 = [by_auc.best_params_["model__C"], by_auc.best_params_["model__C"] == by_ap.best_params_["model__C"]]
```

Both metrics pick a `C` of 1.0 here, so the answer is `[1.0, True]`. The choice of metric can change the winner when the grid separates the settings; on this grid the best two settings differ by less than the spread of the folds, as unit 1 showed, so a metric that agrees is not a sign that the winner is clear.

</details>

## On your own

### Exercise 2: On your own

**Task:** **Capacity of 100.** The retention team can contact 100 people. Before you start, say in one sentence what you expect: a whole number between 0 and 100, well above the 10 that a flat guess would find. Use the tuned model's chances on the test part, flag the 100 people with the highest chances, and put in `answer_2` how many of them churned. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a whole number between 0 and 100.

In [ ]:
_ = check("Exercise 2", "answer_2", 30)

<details>
<summary>Hint</summary>

`np.argsort(chances)[::-1][:100]` gives the positions of the 100 highest chances; `y_test.to_numpy()[positions].sum()` counts the churners among them.

</details>

<details>
<summary>Half-solution</summary>

```python
chances_test = tuned.predict_proba(X_test)[:, 1]
top = np.argsort(chances_test)[::-1][:100]
answer_2 = int(...)
```

</details>

<details>
<summary>Solution</summary>

```python
chances_test = tuned.predict_proba(X_test)[:, 1]
top = np.argsort(chances_test)[::-1][:100]
answer_2 = int(y_test.to_numpy()[top].sum())
```

30 of the 100 people with the highest chances churned, three times the 10 that picking 100 people at random finds on average (the share of churners in the test part is 0.102).

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** **Spot the flaw.** A colleague fits the balanced model on the training rows and estimates how many people of the test part will churn as the sum of its chances: `answer_3 = round(float(balanced_all.predict_proba(X_test)[:, 1].sum()))`. The test part holds a known number of churners, `int(y_test.sum())`. Change the model, not the test part, so that the estimate is close to it, and put the rounded estimate in `answer_3`.

**Expected:** `answer_3` is a whole number.

In [ ]:
balanced_all = build(LogisticRegression(max_iter=1000, class_weight="balanced")).fit(X_train, y_train)
answer_3 = round(float(balanced_all.predict_proba(X_test)[:, 1].sum()))     # colleague: "the chances add up to the churners to expect"
print(answer_3, int(y_test.sum()))
# change the model so that the estimate is close to the number of churners

In [ ]:
_ = check("Exercise 3", "answer_3", 235, traps={1021: "That is the sum of the balanced model's chances. Which model's chances match the share of churners?"}, tol=0.5)

<details>
<summary>Hint</summary>

Which of the two models gives chances that match how often people churn in the rows it learned from?

</details>

<details>
<summary>Half-solution</summary>

```python
plain_all = build(LogisticRegression(max_iter=1000)).fit(X_train, y_train)
answer_3 = round(float(plain_all.predict_proba(X_test)[:, 1].sum()))
```

</details>

<details>
<summary>Solution</summary>

```python
plain_all = build(LogisticRegression(max_iter=1000)).fit(X_train, y_train)
answer_3 = round(float(plain_all.predict_proba(X_test)[:, 1].sum()))
```

The plain model's chances add up to 235, close to the 231 churners of the test part. The balanced model weights each churner about nine times more, so its chances are pushed up and add up to 1,021: good for ranking people and for flagging at a cut, wrong as a chance. A cost rule that needs the real chance of churning needs the plain model's chances, or chances corrected afterwards.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")